In [3]:
import pandas as pd
import sys

sys.path.append("..")

from dados.pacientes import pacientes
from dados.consultas import consultas

In [4]:
df_pacientes = pd.DataFrame(pacientes)
df_consultas = pd.DataFrame(consultas)

print("Pacientes:")
print(df_pacientes.head())
print("\nConsultas:")
print(df_consultas.head())


Pacientes:
   id          nome  idade sexo     cidade    plano
0   1     Ana Silva     34    F  São Paulo  premium
1   2   Carlos Lima     52    M   Campinas   basico
2   3   Julia Souza     28    F  São Paulo  premium
3   4  Marcos Costa     61    M     Santos   basico
4   5   Paula Rocha     45    F   Campinas  premium

Consultas:
   id  id_paciente         medico especialidade  valor     status        data
0   1            1    Dr. Roberto   Cardiologia  350.0  realizada  2026-01-05
1   2            2  Dra. Fernanda     Ortopedia  280.0  realizada  2026-01-06
2   3            3    Dr. Roberto   Cardiologia  350.0  cancelada  2026-01-07
3   4            4  Dra. Patricia    Neurologia  420.0  realizada  2026-01-08
4   5            5  Dra. Fernanda     Ortopedia  280.0  realizada  2026-01-09


In [5]:
print("Shape pacientes:", df_pacientes.shape)
print("Shape consultas:", df_consultas.shape)
print("\nColunas pacientes:", df_pacientes.columns.tolist())
print("Colunas consultas:", df_consultas.columns.tolist())

Shape pacientes: (10, 6)
Shape consultas: (15, 7)

Colunas pacientes: ['id', 'nome', 'idade', 'sexo', 'cidade', 'plano']
Colunas consultas: ['id', 'id_paciente', 'medico', 'especialidade', 'valor', 'status', 'data']


In [6]:
print("=== ANÁLISE EXPLORATÓRIA — PACIENTES ===\n")
print("Total de pacientes:", df_pacientes.shape[0])
print("Colunas:", df_pacientes.columns.tolist())
print()
print("Tipos de dados:")
print(df_pacientes.dtypes)
print()
print("Valores nulos:")
print(df_pacientes.isnull().sum())
print()
print("Estatísticas:")
print(df_pacientes.describe())

=== ANÁLISE EXPLORATÓRIA — PACIENTES ===

Total de pacientes: 10
Colunas: ['id', 'nome', 'idade', 'sexo', 'cidade', 'plano']

Tipos de dados:
id         int64
nome      object
idade      int64
sexo      object
cidade    object
plano     object
dtype: object

Valores nulos:
id        0
nome      0
idade     0
sexo      0
cidade    0
plano     0
dtype: int64

Estatísticas:
             id      idade
count  10.00000  10.000000
mean    5.50000  45.000000
std     3.02765  13.416408
min     1.00000  28.000000
25%     3.25000  35.000000
50%     5.50000  43.000000
75%     7.75000  54.250000
max    10.00000  67.000000


In [7]:
print("=== ANÁLISE EXPLORATÓRIA — CONSULTAS ===\n")
print("Total de consultas:", df_consultas.shape[0])
print()
print("Tipos de dados:")
print(df_consultas.dtypes)
print()
print("Valores nulos:")
print(df_consultas.isnull().sum())
print()
print("Estatísticas:")
print(df_consultas.describe())

=== ANÁLISE EXPLORATÓRIA — CONSULTAS ===

Total de consultas: 15

Tipos de dados:
id                 int64
id_paciente        int64
medico            object
especialidade     object
valor            float64
status            object
data              object
dtype: object

Valores nulos:
id               0
id_paciente      0
medico           0
especialidade    0
valor            0
status           0
data             0
dtype: int64

Estatísticas:
              id  id_paciente       valor
count  15.000000    15.000000   15.000000
mean    8.000000     5.333333  320.000000
std     4.472136     2.968084   81.591316
min     1.000000     1.000000  200.000000
25%     4.500000     3.000000  280.000000
50%     8.000000     5.000000  350.000000
75%    11.500000     7.500000  385.000000
max    15.000000    10.000000  420.000000


In [8]:
print("Cidades únicas:",     df_pacientes["cidade"].unique())
print("Planos únicos:",      df_pacientes["plano"].unique())
print("Especialidades:",     df_consultas["especialidade"].unique())
print("Status consultas:",   df_consultas["status"].unique())
print("Médicos:",            df_consultas["medico"].unique())

Cidades únicas: ['São Paulo' 'Campinas' 'Santos']
Planos únicos: ['premium' 'basico']
Especialidades: ['Cardiologia' 'Ortopedia' 'Neurologia' 'Pediatria']
Status consultas: ['realizada' 'cancelada']
Médicos: ['Dr. Roberto' 'Dra. Fernanda' 'Dra. Patricia' 'Dr. Marcus']


In [9]:
# Quero saber o valor total arrecadado por cada médico, 
# considerando apenas consultas realizadas, ordenado do maior para o menor.


def valor_arrecadado(consultas):
    return consultas[consultas["status"]== 'realizada']\
        .groupby("medico")["valor"].sum().reset_index()\
        .rename(columns={"valor":"valor_total"}) \
        .sort_values("valor_total", ascending=False)

resultado = valor_arrecadado(df_consultas)
print(resultado)

          medico  valor_total
3  Dra. Patricia       1260.0
2  Dra. Fernanda       1120.0
1    Dr. Roberto       1050.0
0     Dr. Marcus        400.0


In [10]:
def consultas_por_especialidade(consultas):
    return consultas.groupby("especialidade")["id"].count().reset_index() \
                    .rename(columns={"id": "quantidade"}) \
                    .sort_values("quantidade", ascending=False)

resultado = consultas_por_especialidade(df_consultas)
print(resultado)

  especialidade  quantidade
0   Cardiologia           4
1    Neurologia           4
2     Ortopedia           4
3     Pediatria           3


In [11]:
def pacientes_por_plano(pacientes):
    return pacientes.groupby("plano").agg(
        quantidade = ("id","count"),
        media_idade =("idade", "mean")
        ).reset_index() \
         .sort_values("media_idade",ascending=False)

resultado = pacientes_por_plano(df_pacientes)
print(resultado)

     plano  quantidade  media_idade
0   basico           5         49.4
1  premium           5         40.6


In [12]:
def consultas_realizadas(consultas):
    return consultas[consultas["status"] == "realizada"] \
        .groupby("especialidade").agg(
            contagem_consulta=("id_paciente", "count"),
            valor_total=("valor", "sum")
        ).reset_index() \
        .sort_values("valor_total", ascending=False)

resultado = consultas_realizadas(df_consultas)
print(resultado)

  especialidade  contagem_consulta  valor_total
1    Neurologia                  3       1260.0
2     Ortopedia                  4       1120.0
0   Cardiologia                  3       1050.0
3     Pediatria                  2        400.0


In [13]:
print("Cidades únicas:",     df_pacientes["cidade"].unique())
print("Planos únicos:",      df_pacientes["plano"].unique())
print("Especialidades:",     df_consultas["especialidade"].unique())
print("Status consultas:",   df_consultas["status"].unique())
print("Médicos:",            df_consultas["medico"].unique())

Cidades únicas: ['São Paulo' 'Campinas' 'Santos']
Planos únicos: ['premium' 'basico']
Especialidades: ['Cardiologia' 'Ortopedia' 'Neurologia' 'Pediatria']
Status consultas: ['realizada' 'cancelada']
Médicos: ['Dr. Roberto' 'Dra. Fernanda' 'Dra. Patricia' 'Dr. Marcus']


In [14]:
def consulta_por_paciente(consultas, pacientes):
    return consultas.merge(pacientes, left_on="id_paciente", right_on="id") \
        .groupby(["nome", "cidade", "plano"])["id_x"].count().reset_index() \
        .rename(columns={"id_x": "quantidade_consulta"}) \
        .sort_values("quantidade_consulta", ascending=False)

resultado = consulta_por_paciente(df_consultas, df_pacientes)
print(resultado)

           nome     cidade    plano  quantidade_consulta
0     Ana Silva  São Paulo  premium                    2
1   Camila Dias  São Paulo  premium                    2
4   Julia Souza  São Paulo  premium                    2
5  Lucia Ferraz     Santos  premium                    2
7   Paula Rocha   Campinas  premium                    2
2   Carlos Lima   Campinas   basico                    1
3  João Pereira  São Paulo   basico                    1
6  Marcos Costa     Santos   basico                    1
8   Pedro Alves     Santos   basico                    1
9  Rafael Nunes   Campinas   basico                    1


In [15]:
def formulario_consulta(consultas, pacientes):
    df = consultas.merge(pacientes, left_on="id_paciente", right_on="id")
    df = df[df["status"] == "realizada"]
    df = df[["nome", "medico", "especialidade", "valor"]]
    df = df.sort_values("valor", ascending=False)
    return df

resultado = formulario_consulta(df_consultas, df_pacientes)
print(resultado)

            nome         medico especialidade  valor
3   Marcos Costa  Dra. Patricia    Neurologia  420.0
5   João Pereira  Dra. Patricia    Neurologia  420.0
8    Camila Dias  Dra. Patricia    Neurologia  420.0
0      Ana Silva    Dr. Roberto   Cardiologia  350.0
6   Lucia Ferraz    Dr. Roberto   Cardiologia  350.0
13  Lucia Ferraz    Dr. Roberto   Cardiologia  350.0
1    Carlos Lima  Dra. Fernanda     Ortopedia  280.0
4    Paula Rocha  Dra. Fernanda     Ortopedia  280.0
10     Ana Silva  Dra. Fernanda     Ortopedia  280.0
14   Camila Dias  Dra. Fernanda     Ortopedia  280.0
9    Pedro Alves     Dr. Marcus     Pediatria  200.0
11   Julia Souza     Dr. Marcus     Pediatria  200.0


In [16]:
def dados_consulta(consultas, pacientes):
    df = consultas.merge(pacientes, left_on="id_paciente", right_on="id")
    df = df[df["status"] == "realizada"]
    df = df.groupby(["nome", "cidade"])["valor"].sum().reset_index()
    df = df.rename(columns={"valor": "valor_total"})
    df = df.sort_values("valor_total", ascending=False)
    return df

resultado = dados_consulta(df_consultas, df_pacientes)
print(resultado)

           nome     cidade  valor_total
1   Camila Dias  São Paulo        700.0
5  Lucia Ferraz     Santos        700.0
0     Ana Silva  São Paulo        630.0
3  João Pereira  São Paulo        420.0
6  Marcos Costa     Santos        420.0
2   Carlos Lima   Campinas        280.0
7   Paula Rocha   Campinas        280.0
4   Julia Souza  São Paulo        200.0
8   Pedro Alves     Santos        200.0


In [17]:
def dados_consulta(consultas, pacientes):
    df = consultas.merge(pacientes, left_on="id_paciente", right_on="id")[lambda df:df["status"] == "realizada"]
    df = df.groupby(["nome", "cidade"])["valor"].sum().reset_index()
    df = df.rename(columns={"valor": "valor_total"})
    df = df.sort_values("valor_total", ascending=False)
    return df

resultado = dados_consulta(df_consultas, df_pacientes)
print(resultado)

           nome     cidade  valor_total
1   Camila Dias  São Paulo        700.0
5  Lucia Ferraz     Santos        700.0
0     Ana Silva  São Paulo        630.0
3  João Pereira  São Paulo        420.0
6  Marcos Costa     Santos        420.0
2   Carlos Lima   Campinas        280.0
7   Paula Rocha   Campinas        280.0
4   Julia Souza  São Paulo        200.0
8   Pedro Alves     Santos        200.0


In [18]:
def dados_paciente(consultas, pacientes):
    df = consultas.merge(pacientes, left_on="id_paciente", right_on="id") \
                  [lambda df: (df["status"] == "realizada") & (df["plano"] == "premium")]
    df = df[["nome", "plano", "especialidade", "valor"]]
    df = df.sort_values("valor", ascending=False)
    return df

resultado = dados_paciente(df_consultas, df_pacientes)
print(resultado)

            nome    plano especialidade  valor
8    Camila Dias  premium    Neurologia  420.0
0      Ana Silva  premium   Cardiologia  350.0
6   Lucia Ferraz  premium   Cardiologia  350.0
13  Lucia Ferraz  premium   Cardiologia  350.0
4    Paula Rocha  premium     Ortopedia  280.0
10     Ana Silva  premium     Ortopedia  280.0
14   Camila Dias  premium     Ortopedia  280.0
11   Julia Souza  premium     Pediatria  200.0


In [19]:
def consultas_especialidade(consultas,pacientes):
    df = consultas.merge(pacientes, left_on = "id_paciente", right_on="id")\
        [lambda df: (df["plano"] == "premium")]
    df = df.groupby(["nome","especialidade"])["id_paciente"].count().reset_index()
    df = df.rename(columns={"id_paciente":"quantidade_consulta"})
    df = df.sort_values("quantidade_consulta", ascending=False)
    return df.sort_values("quantidade_consulta", ascending=False)
resultado = consultas_especialidade(df_consultas,df_pacientes)
print(resultado)

           nome especialidade  quantidade_consulta
6  Lucia Ferraz   Cardiologia                    2
0     Ana Silva   Cardiologia                    1
1     Ana Silva     Ortopedia                    1
2   Camila Dias    Neurologia                    1
3   Camila Dias     Ortopedia                    1
4   Julia Souza   Cardiologia                    1
5   Julia Souza     Pediatria                    1
7   Paula Rocha    Neurologia                    1
8   Paula Rocha     Ortopedia                    1


In [ ]:
def consultas_acima_40(consultas, pacientes):
    df = consultas.merge(pacientes, left_on="id_paciente", right_on="id") \
                  [lambda df: (df["idade"] > 40) & (df["status"] == "realizada")]
    df = df[["nome", "valor", "medico", "cidade"]]
    df = df.sort_values("idade", ascending=False)
    return df

resultado = consultas_acima_40(df_consultas, df_pacientes)
print(resultado)

In [ ]:
def consultas_medico(consultas):
    df = consultas[consultas["status"] == "cancelada"]
    df = df.groupby("medico")["id_paciente"].count().reset_index()
    df = df.rename(columns={"id_paciente": "quantidade_consulta"})
    df = df.sort_values("quantidade_consulta", ascending=False)
    return df

resultado = consultas_medico(df_consultas)
print(resultado)